# Notebook 5 – Aggregation Features

## What are Aggregation Features?

Aggregation features summarize multiple records or values into a meaningful value such as:

- Count
- Sum
- Mean
- Median
- Minimum
- Maximum
- Standard Deviation

Aggregation features are useful when multiple records belong to the same group, customer, product, transaction, or other entity.

### Examples

- Total customer spending
- Average order value
- Number of purchases
- Maximum transaction amount
- Average product rating

In the Titanic dataset, there is no direct customer or product identifier. Therefore, the `Ticket` column is used as a group identifier to demonstrate group-based aggregation.

### Why are Aggregation Features Useful?

Aggregation features can:

- Summarize group-level behavior
- Capture information that individual rows cannot represent
- Provide additional context to machine learning models
- Represent group size, spending, averages, variation, and other patterns
- Help identify relationships between records belonging to the same group

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("Titanic-Dataset.csv")

df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Understanding Aggregation in the Titanic Dataset

The Titanic dataset contains multiple passengers who may share the same `Ticket`.

For example, several passengers can belong to the same ticket group.

Therefore, we can calculate aggregation features such as:

- Number of passengers sharing a ticket
- Total fare paid by the ticket group
- Average fare within the ticket group
- Median fare within the ticket group
- Minimum fare within the ticket group
- Maximum fare within the ticket group
- Fare standard deviation within the ticket group

These features provide group-level information for each passenger.

In [2]:
df["Ticket"].value_counts().head()

Ticket
347082      7
1601        7
CA. 2343    7
3101295     6
CA 2144     6
Name: count, dtype: int64

## 1. Count Aggregation

### Feature Name
`Ticket_Passenger_Count`

### Source Columns
- `Ticket`

### Logic
Count the number of passengers associated with each ticket.

### Reason
The number of people sharing a ticket can represent the size of the passenger group.

### Business Meaning
In a business dataset, this is similar to the number of purchases made by a customer or the number of transactions associated with an account.

### ML Relevance
Group size can provide additional information that may help a model identify patterns that are not visible from an individual record.

### Leakage Check
The feature does not use the target column. However, in a real train/test workflow, group-level statistics should be calculated carefully using training data to avoid information contamination.

### Final Decision
**Retain**

In [3]:
df["Ticket_Passenger_Count"] = df.groupby("Ticket")["Ticket"].transform("count")

df[["Ticket", "Ticket_Passenger_Count"]].head(10)

,Ticket,Ticket_Passenger_Count
0,A/5 21171,1
1,PC 17599,1
2,STON/O2. 3101282,1
3,113803,2
4,373450,1
5,330877,1
6,17463,1
7,349909,4
8,347742,3
9,237736,2


## 2. Sum Aggregation

### Feature Name
`Ticket_Total_Fare`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Calculate the total fare paid by all passengers sharing the same ticket.

### Reason
The total value associated with a group can provide more information than an individual fare.

### Business Meaning
In an e-commerce dataset, this is similar to total customer spending.

### ML Relevance
Total group value can represent the economic size of a group and may help a model identify useful patterns.

### Leakage Check
The target variable is not used. In production, aggregation must be calculated without using future or test information.

### Final Decision
**Retain**

In [4]:
df["Ticket_Total_Fare"] = df.groupby("Ticket")["Fare"].transform("sum")

df[["Ticket", "Fare", "Ticket_Total_Fare"]].head(10)

,Ticket,Fare,Ticket_Total_Fare
0,A/5 21171,7.2500,7.2500
1,PC 17599,71.2833,71.2833
2,STON/O2. 3101282,7.9250,7.9250
3,113803,53.1000,106.2000
4,373450,8.0500,8.0500
5,330877,8.4583,8.4583
6,17463,51.8625,51.8625
7,349909,21.0750,84.3000
8,347742,11.1333,33.3999
9,237736,30.0708,60.1416


## 3. Mean Aggregation

### Feature Name
`Ticket_Average_Fare`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Calculate the average fare for passengers sharing the same ticket.

### Reason
The average provides a normalized group-level measure.

### Business Meaning
In an e-commerce dataset, this is similar to average order value.

### ML Relevance
Average group values can provide context and help the model identify differences between groups.

### Leakage Check
The target is not used. The aggregation should still be calculated in a leakage-safe manner after train/test splitting.

### Final Decision
**Retain**

In [5]:
df["Ticket_Average_Fare"] = df.groupby("Ticket")["Fare"].transform("mean")

df[["Ticket", "Fare", "Ticket_Average_Fare"]].head(10)

,Ticket,Fare,Ticket_Average_Fare
0,A/5 21171,7.2500,7.2500
1,PC 17599,71.2833,71.2833
2,STON/O2. 3101282,7.9250,7.9250
3,113803,53.1000,53.1000
4,373450,8.0500,8.0500
5,330877,8.4583,8.4583
6,17463,51.8625,51.8625
7,349909,21.0750,21.0750
8,347742,11.1333,11.1333
9,237736,30.0708,30.0708


## 4. Median Aggregation

### Feature Name
`Ticket_Median_Fare`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Calculate the median fare for each ticket group.

### Reason
Median is less affected by extreme values than the mean.

### Business Meaning
In a financial or e-commerce dataset, median spending can represent the typical value of a group without being strongly affected by unusually large transactions.

### ML Relevance
Median aggregation can provide a robust measure of the typical group value.

### Leakage Check
The target is not used. The calculation should be performed using training information only when building an ML model.

### Final Decision
**Retain**

In [6]:
df["Ticket_Median_Fare"] = df.groupby("Ticket")["Fare"].transform("median")

df[["Ticket", "Fare", "Ticket_Median_Fare"]].head(10)

,Ticket,Fare,Ticket_Median_Fare
0,A/5 21171,7.2500,7.2500
1,PC 17599,71.2833,71.2833
2,STON/O2. 3101282,7.9250,7.9250
3,113803,53.1000,53.1000
4,373450,8.0500,8.0500
5,330877,8.4583,8.4583
6,17463,51.8625,51.8625
7,349909,21.0750,21.0750
8,347742,11.1333,11.1333
9,237736,30.0708,30.0708


## 5. Minimum Aggregation

### Feature Name
`Ticket_Min_Fare`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Find the minimum fare within each ticket group.

### Reason
It provides information about the lowest fare associated with the group.

### Business Meaning
In a transaction dataset, this could represent the minimum transaction amount for a customer.

### ML Relevance
Minimum values can help describe the lower boundary of group behavior.

### Leakage Check
The target is not used. The feature should be calculated using leakage-safe training data.

### Final Decision
**Retain**

In [7]:
df["Ticket_Min_Fare"] = df.groupby("Ticket")["Fare"].transform("min")

df[["Ticket", "Fare", "Ticket_Min_Fare"]].head(10)

,Ticket,Fare,Ticket_Min_Fare
0,A/5 21171,7.2500,7.2500
1,PC 17599,71.2833,71.2833
2,STON/O2. 3101282,7.9250,7.9250
3,113803,53.1000,53.1000
4,373450,8.0500,8.0500
5,330877,8.4583,8.4583
6,17463,51.8625,51.8625
7,349909,21.0750,21.0750
8,347742,11.1333,11.1333
9,237736,30.0708,30.0708


## 6. Maximum Aggregation

### Feature Name
`Ticket_Max_Fare`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Find the maximum fare within each ticket group.

### Reason
It provides information about the highest fare associated with the group.

### Business Meaning
In a transaction dataset, this could represent the maximum transaction amount for a customer.

### ML Relevance
Maximum values can capture extreme group-level behavior that may be useful to a model.

### Leakage Check
The target is not used. The aggregation must be calculated without using information from the validation or test set.

### Final Decision
**Retain**

In [8]:
df["Ticket_Max_Fare"] = df.groupby("Ticket")["Fare"].transform("max")

df[["Ticket", "Fare", "Ticket_Max_Fare"]].head(10)

,Ticket,Fare,Ticket_Max_Fare
0,A/5 21171,7.2500,7.2500
1,PC 17599,71.2833,71.2833
2,STON/O2. 3101282,7.9250,7.9250
3,113803,53.1000,53.1000
4,373450,8.0500,8.0500
5,330877,8.4583,8.4583
6,17463,51.8625,51.8625
7,349909,21.0750,21.0750
8,347742,11.1333,11.1333
9,237736,30.0708,30.0708


## 7. Standard Deviation Aggregation

### Feature Name
`Ticket_Fare_Std`

### Source Columns
- `Ticket`
- `Fare`

### Logic
Calculate the standard deviation of fare within each ticket group.

### Reason
Standard deviation measures how much the fare values vary within the group.

### Business Meaning
In a business dataset, this can represent variation in customer spending or transaction values.

### ML Relevance
Variation can provide additional information about group behavior that cannot be captured by the average alone.

### Leakage Check
The target is not used. The aggregation should be generated from appropriate training data during model development.

### Final Decision
**Retain**

In [9]:
df["Ticket_Fare_Std"] = df.groupby("Ticket")["Fare"].transform("std")

df["Ticket_Fare_Std"] = df["Ticket_Fare_Std"].fillna(0)

df[["Ticket", "Fare", "Ticket_Fare_Std"]].head(10)

,Ticket,Fare,Ticket_Fare_Std
0,A/5 21171,7.2500,0.0
1,PC 17599,71.2833,0.0
2,STON/O2. 3101282,7.9250,0.0
3,113803,53.1000,0.0
4,373450,8.0500,0.0
5,330877,8.4583,0.0
6,17463,51.8625,0.0
7,349909,21.0750,0.0
8,347742,11.1333,0.0
9,237736,30.0708,0.0


## 8. Group-Based Aggregation

Group-based aggregation means calculating statistics for records that belong to the same group.

In the Titanic dataset, `Ticket` is used as the grouping column.

The following group-level features have been created:

- `Ticket_Passenger_Count`
- `Ticket_Total_Fare`
- `Ticket_Average_Fare`
- `Ticket_Median_Fare`
- `Ticket_Min_Fare`
- `Ticket_Max_Fare`
- `Ticket_Fare_Std`

These features give each passenger additional information about their ticket group.

In [10]:
aggregation_features = [
    "Ticket_Passenger_Count",
    "Ticket_Total_Fare",
    "Ticket_Average_Fare",
    "Ticket_Median_Fare",
    "Ticket_Min_Fare",
    "Ticket_Max_Fare",
    "Ticket_Fare_Std"
]

df[["Ticket"] + aggregation_features].head(10)

,Ticket,Ticket_Passenger_Count,Ticket_Total_Fare,Ticket_Average_Fare,Ticket_Median_Fare,Ticket_Min_Fare,Ticket_Max_Fare,Ticket_Fare_Std
0,A/5 21171,1,7.2500,7.2500,7.2500,7.2500,7.2500,0.0
1,PC 17599,1,71.2833,71.2833,71.2833,71.2833,71.2833,0.0
2,STON/O2. 3101282,1,7.9250,7.9250,7.9250,7.9250,7.9250,0.0
3,113803,2,106.2000,53.1000,53.1000,53.1000,53.1000,0.0
4,373450,1,8.0500,8.0500,8.0500,8.0500,8.0500,0.0
5,330877,1,8.4583,8.4583,8.4583,8.4583,8.4583,0.0
6,17463,1,51.8625,51.8625,51.8625,51.8625,51.8625,0.0
7,349909,4,84.3000,21.0750,21.0750,21.0750,21.0750,0.0
8,347742,3,33.3999,11.1333,11.1333,11.1333,11.1333,0.0
9,237736,2,60.1416,30.0708,30.0708,30.0708,30.0708,0.0


## 9. Customer-Level Aggregation

Customer-level aggregation is commonly used when multiple records belong to the same customer.

### Examples

For an e-commerce dataset:

- Total customer spending
- Average order value
- Number of purchases
- Maximum transaction amount
- Minimum transaction amount
- Standard deviation of transaction amount

### Example

If a customer has five purchases:

- Count → 5 purchases
- Sum → total spending
- Mean → average order value
- Median → typical order value
- Maximum → largest purchase
- Minimum → smallest purchase
- Standard deviation → variation in purchase values

The Titanic dataset does not contain a customer identifier, so customer-level aggregation is explained as a general business use case rather than being represented as an actual Titanic feature.

In [11]:
# General example of customer-level aggregation

customer_example = pd.DataFrame({
    "CustomerID": [1, 1, 1, 2, 2],
    "PurchaseAmount": [100, 150, 200, 80, 120]
})

customer_aggregation = customer_example.groupby("CustomerID")["PurchaseAmount"].agg(
    Total_Spending="sum",
    Average_Order_Value="mean",
    Number_of_Purchases="count",
    Maximum_Purchase="max",
    Minimum_Purchase="min",
    Spending_Std="std"
).reset_index()

customer_aggregation

,CustomerID,Total_Spending,Average_Order_Value,Number_of_Purchases,Maximum_Purchase,Minimum_Purchase,Spending_Std
0,1,450,150.0,3,200,100,50.000000
1,2,200,100.0,2,120,80,28.284271


## 10. Product-Level Aggregation

Product-level aggregation summarizes information for each product.

### Examples

For an e-commerce dataset:

- Number of purchases for a product
- Total product sales
- Average product rating
- Maximum product price
- Minimum product price
- Standard deviation of product ratings

### Why is it useful?

Product-level features can help a machine learning model understand product popularity, sales performance, customer interaction, and variation in product-related behavior.

The Titanic dataset does not contain product information, so this is demonstrated as a general business example.

In [12]:
product_example = pd.DataFrame({
    "ProductID": ["A", "A", "B", "B", "B"],
    "Rating": [4, 5, 3, 4, 5],
    "Sales": [100, 150, 80, 120, 200]
})

product_aggregation = product_example.groupby("ProductID").agg(
    Average_Rating=("Rating", "mean"),
    Total_Sales=("Sales", "sum"),
    Maximum_Sales=("Sales", "max"),
    Number_of_Sales=("Sales", "count")
).reset_index()

product_aggregation

,ProductID,Average_Rating,Total_Sales,Maximum_Sales,Number_of_Sales
0,A,4.5,250,150,2
1,B,4.0,400,200,3


## 11. Transaction-Level Aggregation

Transaction-level aggregation is useful when individual transactions contain multiple related values.

Examples include:

- Total transaction amount
- Number of items in a transaction
- Average item price
- Maximum item price
- Minimum item price

Transaction-level aggregation can convert detailed transaction information into compact features that represent transaction behavior.

These features can help ML models identify patterns such as high-value transactions, frequent transactions, or unusual transaction behavior.

The Titanic dataset does not contain transaction records, so this concept is demonstrated using a general example.

In [13]:
transaction_example = pd.DataFrame({
    "TransactionID": [101, 101, 102, 102, 102],
    "ItemPrice": [100, 50, 200, 80, 120]
})

transaction_aggregation = transaction_example.groupby("TransactionID").agg(
    Total_Amount=("ItemPrice", "sum"),
    Average_Item_Price=("ItemPrice", "mean"),
    Maximum_Item_Price=("ItemPrice", "max"),
    Minimum_Item_Price=("ItemPrice", "min"),
    Number_of_Items=("ItemPrice", "count")
).reset_index()

transaction_aggregation

,TransactionID,Total_Amount,Average_Item_Price,Maximum_Item_Price,Minimum_Item_Price,Number_of_Items
0,101,150,75.000000,100,50,2
1,102,400,133.333333,200,80,3


## Aggregation Feature Summary

Aggregation features summarize information at different levels.

| Aggregation | Meaning | Example |
|---|---|---|
| Count | Number of records | Number of purchases |
| Sum | Total value | Total customer spending |
| Mean | Average value | Average order value |
| Median | Middle value | Median transaction amount |
| Minimum | Smallest value | Minimum transaction amount |
| Maximum | Largest value | Maximum transaction amount |
| Standard Deviation | Variation | Spending variation |

### Group-Based Aggregation

Group-based aggregation is especially useful when multiple records belong to the same entity.

Examples:

- Customer-level
- Product-level
- Transaction-level
- Account-level
- Ticket-level

In [14]:
summary = pd.DataFrame({
    "Feature": [
        "Ticket_Passenger_Count",
        "Ticket_Total_Fare",
        "Ticket_Average_Fare",
        "Ticket_Median_Fare",
        "Ticket_Min_Fare",
        "Ticket_Max_Fare",
        "Ticket_Fare_Std"
    ],
    "Aggregation": [
        "Count",
        "Sum",
        "Mean",
        "Median",
        "Minimum",
        "Maximum",
        "Standard Deviation"
    ]
})

summary

,Feature,Aggregation
0,Ticket_Passenger_Count,Count
1,Ticket_Total_Fare,Sum
2,Ticket_Average_Fare,Mean
3,Ticket_Median_Fare,Median
4,Ticket_Min_Fare,Minimum
5,Ticket_Max_Fare,Maximum
6,Ticket_Fare_Std,Standard Deviation


## Final Feature Documentation

The following documentation records the important engineered aggregation features.

| Feature Name | Source Columns | Logic | Reason | Business Meaning | ML Relevance | Leakage Check | Final Decision |
|---|---|---|---|---|---|---|---|
| Ticket_Passenger_Count | Ticket | Count passengers per ticket | Capture group size | Number of passengers in group | Provides group-size information | Target not used; calculate safely after split | Retain |
| Ticket_Total_Fare | Ticket, Fare | Sum Fare by Ticket | Capture group spending | Total fare of ticket group | Represents group economic value | Target not used; avoid test contamination | Retain |
| Ticket_Average_Fare | Ticket, Fare | Mean Fare by Ticket | Capture typical group fare | Average fare of group | Provides group-level context | Target not used; train-only aggregation recommended | Retain |
| Ticket_Median_Fare | Ticket, Fare | Median Fare by Ticket | Reduce influence of extreme values | Typical group fare | Robust group-level feature | Target not used | Retain |
| Ticket_Min_Fare | Ticket, Fare | Minimum Fare by Ticket | Capture lower group value | Lowest fare in group | Represents lower boundary | Target not used | Retain |
| Ticket_Max_Fare | Ticket, Fare | Maximum Fare by Ticket | Capture highest group value | Highest fare in group | Represents upper boundary | Target not used | Retain |
| Ticket_Fare_Std | Ticket, Fare | Standard deviation of Fare | Measure variation | Fare variation within group | Captures group-level variability | Target not used | Retain |

### Leakage Consideration

Aggregation features can cause leakage if information from the target, future records, validation data, or test data is included while creating the feature.

For example, a target-based aggregation such as average survival rate by ticket must not be calculated using the complete dataset before splitting.

For safe ML workflows, aggregation statistics should be calculated using training data and then applied to validation or test data.

### Final Decision

The aggregation features created in this notebook are **retained** because they provide additional group-level information without directly using the target variable.

In [15]:
documentation = pd.DataFrame({
    "Feature Name": [
        "Ticket_Passenger_Count",
        "Ticket_Total_Fare",
        "Ticket_Average_Fare",
        "Ticket_Median_Fare",
        "Ticket_Min_Fare",
        "Ticket_Max_Fare",
        "Ticket_Fare_Std"
    ],
    "Source Columns": [
        "Ticket",
        "Ticket, Fare",
        "Ticket, Fare",
        "Ticket, Fare",
        "Ticket, Fare",
        "Ticket, Fare",
        "Ticket, Fare"
    ],
    "Leakage Risk": [
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low",
        "Low"
    ],
    "Final Decision": [
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Retain",
        "Retain"
    ]
})

documentation

,Feature Name,Source Columns,Leakage Risk,Final Decision
0,Ticket_Passenger_Count,Ticket,Low,Retain
1,Ticket_Total_Fare,"Ticket, Fare",Low,Retain
2,Ticket_Average_Fare,"Ticket, Fare",Low,Retain
3,Ticket_Median_Fare,"Ticket, Fare",Low,Retain
4,Ticket_Min_Fare,"Ticket, Fare",Low,Retain
5,Ticket_Max_Fare,"Ticket, Fare",Low,Retain
6,Ticket_Fare_Std,"Ticket, Fare",Low,Retain


## Conclusion

Aggregation features convert multiple records into meaningful summary information.

In this notebook, aggregation techniques including count, sum, mean, median, minimum, maximum, and standard deviation were demonstrated.

Using the Titanic dataset, ticket-level aggregation was used to create group-based features such as passenger count and fare statistics.

Aggregation features are particularly useful in business datasets because they can represent customer behavior, product performance, and transaction patterns.

However, aggregation must be designed carefully to avoid data leakage. In a real machine learning workflow, aggregation statistics should be calculated using appropriate training data before being applied to validation or test data.